In [2]:
!pip install -q torch torchvision \transformers \tqdm \pillow \ boto3

In [3]:
import json
import logging
import os
from transformers import ViltProcessor

from data.clevr_curriculum_data import (
    CLEVRCurriculumViltDatasetS3,
    vilt_collate_fn,
)
from models.vilt_adapter import ViLTAdapter
from services.checkpoint_service import CheckpointManager
from training.curriculum_trainer import CurriculumTrainer

In [4]:
CONFIG = {
    # S3
    "s3_bucket":           "clevr-curriculum",
    "s3_images_prefix":    "dataset/images",
    "s3_questions_prefix": "dataset/questions",

    "answer_vocab_path": "data/answer_vocab.json",

    # Dataset
    "tiers":               [1, 2, 3, 4, 5],
    "max_question_length": 32,

    # Model
    "model_name":          "dandelin/vilt-b32-finetuned-vqa",
    "learning_rate":       5e-5,
    "freeze_backbone":     False,
    "device":              "cuda",   

    # Curriculum Configurations
    "beta":                0.85,      # EMA smoothing factor
    "entropy_weight":      0.7,      # alpha in competence calculation
    "loss_weight":         0.3,
    "c_min":               0.05,     # competence clamp lower
    "c_max":               0.99,     # competence clamp upper

    # Tier difficulty exponents 
    "difficulty": {
        1: 1.0,   # Attribute & Existence
        2: 2.25,   # Compare Attribute
        3: 3.5,   # Counting / Compare Integer
        4: 5.0,   # Relational Tasks
        5: 6.5,  # Complex Composition
    },

    # Soft Self-Paced Learning 
    "spl_lambda_init":     0.5,      # SPL temperature at low competence
    "spl_lambda_max":      5.0,      # SPL temperature at high competence

    # Training Configurations
    "batch_size":          32,
    "num_steps":           100,
    "val_every":           100,      # validate every N steps
    "checkpoint_every":    50,      # checkpoint to S3 every N steps
    "log_every":           50,       # print log every N steps

    # Checkpointing
    "run_name":            "curriculum_run_test1",
    "checkpoint_prefix":   "checkpoints",
    "resume":              True,    
}

In [5]:
# logging
def _setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(name)-28s  %(levelname)-7s  %(message)s"
    )

def load_answer_vocab(config: dict) -> dict:
    """
    Load the answer -> id mapping from a  JSON file. eg: {"yes": 27}
    """
    path = config["answer_vocab_path"]
    with open(path, "r") as f:
        answer2id = json.load(f)
    logging.info("Loaded answer vocab from %s  (%d classes)", path, len(answer2id))
    return answer2id

def build_datasets(config: dict, processor, answer2id: dict):
    """
    Create one training dataset per tier and one combined validation dataset.
    All data is streamed from S3.
    """
    tier_datasets = {}
    for t in config["tiers"]:
        tier_datasets[t] = CLEVRCurriculumViltDatasetS3(
            bucket=config["s3_bucket"],
            images_prefix=config["s3_images_prefix"],
            questions_prefix=config["s3_questions_prefix"],
            processor=processor,
            split="train",
            answer2id=answer2id,
            tiers=[t],
            max_length=config["max_question_length"],
        )
        logging.info("Tier %d training set: %d samples", t, len(tier_datasets[t]))

    val_dataset = CLEVRCurriculumViltDatasetS3(
        bucket=config["s3_bucket"],
        images_prefix=config["s3_images_prefix"],
        questions_prefix=config["s3_questions_prefix"],
        processor=processor,
        split="val",
        answer2id=answer2id,
        tiers=config["tiers"],
        max_length=config["max_question_length"],
    )
    logging.info("Validation set: %d samples", len(val_dataset))

    # Per-tier validation datasets
    tier_val_datasets = {}
    for t in config["tiers"]:
        tier_val_datasets[t] = CLEVRCurriculumViltDatasetS3(
            bucket=config["s3_bucket"],
            images_prefix=config["s3_images_prefix"],
            questions_prefix=config["s3_questions_prefix"],
            processor=processor,
            split="val",
            answer2id=answer2id,
            tiers=[t],
            max_length=config["max_question_length"],
        )
        logging.info("Tier %d validation set: %d samples", t, len(tier_val_datasets[t]))

    return tier_datasets, val_dataset, tier_val_datasets


In [6]:
def main():
    """
    Entry point for Curriculum Training.

    Execution order:
      1. Config setup
      2. Processor and answer vocab loading
      3. Tiered dataset construction (train tiers and val splits)
      4. ViLTAdapter, model and optimizer initialisation
      5. CheckpointManager setup
      6. CurriculumTrainer assembly with all hyperparameters
      7. Optional resume from the latest S3 checkpoint
      8. Training loop execution and history logging
    """
    config = CONFIG
    _setup_logging()
    logger = logging.getLogger("main")

    logger.info("=" * 60)
    logger.info("  Competence-Aware Curriculum Training")
    logger.info("=" * 60)

    # Load Processor and Answer Vocab
    processor = ViltProcessor.from_pretrained(config["model_name"])
    answer2id = load_answer_vocab(config)
    num_classes = len(answer2id)

    # Load Datasets
    tier_datasets, val_dataset, tier_val_datasets = build_datasets(config, processor, answer2id)

    # Model
    model = ViLTAdapter(
        model_name=config["model_name"],
        num_labels=num_classes,
        learning_rate=config["learning_rate"],
        device=config["device"],
        freeze_backbone=config["freeze_backbone"],
    )
    logger.info("Model loaded: %s  (num_labels=%d)", config["model_name"], num_classes)

    #  Checkpoint manager (S3)
    ckpt = CheckpointManager(
        bucket=config["s3_bucket"],
        run_name=config["run_name"],
        prefix=config["checkpoint_prefix"],
    )

    # Curriculum Trainer
    trainer = CurriculumTrainer(
        model=model,
        tier_datasets=tier_datasets,
        val_dataset=val_dataset,
        num_classes=num_classes,
        checkpoint_manager=ckpt,
        batch_size=config["batch_size"],
        num_steps=config["num_steps"],
        beta=config["beta"],
        entropy_weight=config["entropy_weight"],
        loss_weight=config["loss_weight"],
        collate_fn=vilt_collate_fn,
        difficulty=config["difficulty"],
        val_every=config["val_every"],
        checkpoint_every=config["checkpoint_every"],
        log_every=config["log_every"],
        tier_val_datasets=tier_val_datasets,
    )

    #  Resume from checkpoint if available
    start_step = 0
    if config["resume"]:
        start_step = trainer.resume(tag="latest")
        if start_step > 0:
            logger.info("Resumed from step %d", start_step)
        else:
            logger.info("No checkpoint found- starting from scratch")

    #  Train
    logger.info("Training for %d steps (starting at step %d)", config["num_steps"], start_step)
    history = trainer.train(start_step=start_step)

    logger.info("=" * 60)
    logger.info("  Training finished  |  %d steps completed", len(history))
    logger.info("=" * 60)

In [ ]:

if __name__ == "__main__":
    main()

main                          INFO     ============================================================
main                          INFO       Competence-Aware Curriculum Training
main                          INFO     ============================================================
httpx                         INFO     HTTP Request: GET https://huggingface.co/api/models/dandelin/vilt-b32-finetuned-vqa/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Fo

preprocessor_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dandelin/vilt-b32-finetuned-vqa/d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08/preprocessor_config.json "HTTP/1.1 200 OK"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dandelin/vilt-b32-finetuned-vqa/d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08/config.json "HTTP/1.1 200 OK"
httpx                         INFO     

config.json: 0.00B [00:00, ?B/s]

httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx                         INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dandelin/vilt-b32-finetuned-vqa/d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08/tokenizer_config.json "HTTP/1.1 200 OK"
httpx                         INFO     HTTP Request: GET https://huggingface.co/api/resolve-cache/models/dandelin/vilt-b32-finetuned-vqa/d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08/tokenizer_config.json "HTTP/1.1 200 OK"


...

httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
httpx                         INFO     HTTP Request: GET https://huggingface.co/api/models/dandelin/vilt-b32-finetuned-vqa "HTTP/1.1 200 OK"
httpx                         INFO     HTTP Request: GET https://huggingface.co/api/models/dandelin/vilt-b32-finetuned-vqa/commits/main "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

ViltForQuestionAnswering LOAD REPORT from: dandelin/vilt-b32-finetuned-vqa
Key                                          | Status     |                                                                                             
---------------------------------------------+------------+---------------------------------------------------------------------------------------------
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |                                                                                             
classifier.3.weight                          | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3129, 1536]) vs model:torch.Size([28, 1536])
classifier.3.bias                            | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3129]) vs model:torch.Size([28])            

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH:	ckpt weights were loaded, b

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

main                          INFO     Model loaded: dandelin/vilt-b32-finetuned-vqa  (num_labels=28)
